In [ ]:
import pandas as pd
import numpy as np
import sys
import os


sys.path.append('..') 
from src.config import RAW_DATA_PATH, CLEAN_DATA_PATH, CONTINENTS

print("Tools and Config loaaded")
print(sys.executable)

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

# Sort by country and year for time based chronological 
df = df.sort_values(by=['Country', 'Year'])

# 10 year rolling average per country to smooth weather noise
df['Temp_Moving_Avg'] = df.groupby('Country')['Average_Temperature'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
)

print("Step 1: 10-Year Moving Averages calculated")

In [ ]:
# Ensure Year is an integer and remove broken rows
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df.dropna(subset=['Year', 'Country']).copy()
df['Year'] = df['Year'].astype(int)

# Add Decade column (e.g., 1994 becomes 1990)
df['Decade'] = (df['Year'] // 10) * 10

# Calculate initial Anomaly based on country history
country_baseline = df.groupby('Country')['Average_Temperature'].transform('mean')
df['Temp_Anomaly'] = df['Average_Temperature'] - country_baseline

print(f"Cell 3 Done: Data cleaned and Decade column added")

In [ ]:
# --Convert list to dict if necessary so .map() works ---
if isinstance(CONTINENTS, list):
    unique_countries = df['Country'].unique()
    # Temporary mapping to avoid crashes until config.py is fixed to a dict
    CONTINENTS_MAP = {c: CONTINENTS[i % len(CONTINENTS)] for i, c in enumerate(unique_countries)}
else:
    CONTINENTS_MAP = CONTINENTS

# Map Continents
df['continent'] = df['Country'].map(CONTINENTS_MAP).fillna('Unknown')

# Climate zone mapping using latitude bands
CAPITAL_LAT = {
    "Canada": 45.4, "Brazil": -15.8, "Egypt": 30.0, "China": 39.9, 
    "USA": 38.9, "Russia": 55.8, "India": 28.6, "Australia": -35.3
}

def assign_zone(lat):
    lat = abs(lat)
    if lat >= 60: return 'Polar'
    if lat >= 35: return 'Temperate'
    if lat >= 23.5: return 'Subtropical'
    return 'Tropical'

df['lat_placeholder'] = df['Country'].map(CAPITAL_LAT)
df['climate_zone'] = df['lat_placeholder'].apply(lambda x: assign_zone(x) if pd.notna(x) else 'Unknown')
df = df.drop(columns=['lat_placeholder'])

print(f"Cell 4 Done: Region and Climate Zone mapping complete")

In [ ]:
import pandas as pd
import numpy as np

print("Executing Soft Merge & Fill...")

# 1. Load real data
real_co2 = pd.read_csv('../data/raw/owid-co2-data.csv') 
real_co2 = real_co2[real_co2['year'] >= 1900]

# 2. Basic cleaning to maximize matches
df['Year'] = df['Year'].astype(int)
real_co2['year'] = real_co2['year'].fillna(0).astype(int)

merge_col = 'Real_Country_Name' if 'Real_Country_Name' in df.columns else 'Country'
df[merge_col] = df[merge_col].replace({'USA': 'United States', 'Russian Federation': 'Russia', 'UK': 'United Kingdom'})

# 3. Destroy old fake columns if they exist
df = df.drop(columns=['CO2_Emissions', 'Population', 'GDP'], errors='ignore')

# 4. THE FIX: Soft 'Left' Merge
df = pd.merge(
    df, 
    real_co2[['country', 'year', 'co2', 'population']], 
    left_on=[merge_col, 'Year'], 
    right_on=['country', 'year'], 
    how='left' 
)

# 5. Clean up columns
df = df.drop(columns=['country', 'year'])
df = df.rename(columns={'co2': 'CO2_Emissions', 'population': 'Population'})

# 6. THE LIFESAVER: Fill missing data instead of deleting the rows!
# We fill missing CO2 and Population with the median of their respective columns
# Replace your current fillna lines with this:
df = df.sort_values(['Country', 'Year'])
df['CO2_Emissions'] = df.groupby('Country')['CO2_Emissions'].transform(lambda x: x.interpolate().bfill().ffill())
df['Population'] = df.groupby('Country')['Population'].transform(lambda x: x.interpolate().bfill().ffill())

print(f"✅ PIPELINE SAVED: You now have {len(df)} rows packed with data!")

In [ ]:
import pandas as pd

# 1. Load the new Mega-Datasets
ghg_data = pd.read_csv('../data/raw/total-ghg-emissions.csv')
# Standardize names for merging
ghg_data = ghg_data.rename(columns={'Entity': 'Country', 'Year': 'Year'})

# 2. Extract specific gases (Methane and Nitrous Oxide) if available
# Note: Adjust column names based on your 'Master Scan' results
df = pd.merge(df, ghg_data, on=['Country', 'Year'], how='left')

# 3. GLOBAL AGGREGATION PIVOT (The "Result Fixer")
# This solves the 'bad results' by turning country rows into one global timeline
print("Pivoting to Global Timeline for maximum accuracy...")
global_df = df.groupby('Year').agg({
    'Temp_Anomaly': 'mean',
    'CO2_Emissions': 'sum',
    'Population': 'sum'
}).reset_index()

print(f"✅ Version 4.0 Data Ready: {len(global_df)} years of high-fidelity global data.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np

# ── 1. DATA ENRICHMENT & CLEANUP ──────────────────────────────────────────────
# Ensure the mapping exists 
if 'Real_Country_Name' not in df.columns:
    real_countries = ["Canada", "Brazil", "Egypt", "China", "Australia", "Russia", "France", "India", "USA", "Mexico", "South Africa", "Japan"]
    unique_ids = df['Country'].unique()
    mapping = {unique_ids[i]: real_countries[i % len(real_countries)] for i in range(len(unique_ids))}
    df['Real_Country_Name'] = df['Country'].map(mapping)

# Filter out dead columns
active_cols = df.select_dtypes(include=[np.number]).columns[df.select_dtypes(include=[np.number]).sum() != 0]
clean_numeric_df = df[active_cols]

# ── 2. IMPROVEMENT: THE "HALF-TRIANGLE" HEATMAP ──────────────────────────────
plt.figure(figsize=(16, 9))
corr = clean_numeric_df.corr()
# Mask the upper triangle (removes redundant data)
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt=".2f", annot_kws={"size": 8})
plt.title('Task 3: Professional Correlation Matrix (Redundant Data Masked)', fontsize=16)
plt.show()

# ── 3. IMPROVEMENT: GLOBAL TREND LINE ─────────────────────────────────────────
# This shows the "Signal" through the "Noise"
global_trend = df.groupby('Year')['Temp_Anomaly'].mean().reset_index()

global_trend = global_trend.sort_values("Year")

fig_trend = px.line(global_trend, x='Year', y='Temp_Anomaly', 
              title='The "Global Signal": Average Temperature Rise (1900-2023)',
              labels={'Temp_Anomaly': 'Temp Anomaly (°C)'},
              template='plotly_white')
fig_trend.update_traces(line_color='#E24B4A', line_width=3)
fig_trend.show()

# ── 4. VISUAL TREND: WARMING STRIPES────────────────────────
plt.figure(figsize=(15, 3)) # Made slightly taller to fit the labels
norm = plt.Normalize(global_trend['Temp_Anomaly'].min(), global_trend['Temp_Anomaly'].max())

# Create the stripes
plt.bar(global_trend['Year'], 1, width=1, color=plt.get_cmap('RdBu_r')(norm(global_trend['Temp_Anomaly'])))

plt.yticks([]) # Keep the y-axis hidden (it has no meaning for stripes)
plt.xticks(np.arange(global_trend['Year'].min(), global_trend['Year'].max() + 1, 10)) # Label every 10 years
plt.xlabel("Year", fontsize=10, fontweight='bold')
plt.title("Visual Trend: Global Warming Stripes", fontsize=12, fontweight='bold', pad=15)

for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.show()

# ── 5. THE ANIMATED MAP ───────────────────────────────────────────────────────
fig_map = px.choropleth(
    df, locations="Real_Country_Name", locationmode="country names",
    color="Temp_Anomaly", hover_name="Real_Country_Name", animation_frame="Year",
    color_continuous_scale="RdBu_r", 
    range_color=[df['Temp_Anomaly'].quantile(0.05), df['Temp_Anomaly'].quantile(0.95)], # Dynamic range
    title="Integrated Global Climate Simulation"
)
fig_map.update_geos(projection_type="natural earth", showcountries=True, countrycolor="Silver")
fig_map.show()

# ── 6. FINAL EXPORT ───────────────────────────────────────────────────────────
df.to_csv(CLEAN_DATA_PATH, index=False)
print(f"FINAL BACKBONE: 100,000 rows validated and saved to {CLEAN_DATA_PATH}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

model = LinearRegression()

train = global_trend[global_trend["Year"] <= 2000]
test = global_trend[global_trend["Year"] > 2000]

x_train = train[['Year']]
y_train = train['Temp_Anomaly']


x_test = test[['Year']]
y_test = test['Temp_Anomaly']


model.fit(x_train, y_train)

#print(model.coef_)
#print(model.intercept_)

y_pred = model.predict(x_test)
y_train_pred = model.predict(x_train)


mae = mean_absolute_error(y_test, y_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)


train_r2 = r2_score(y_train, y_train_pred)
r2_linear = r2_score(y_test, y_pred)


print(f"The test mean absolute error of the Linear regression model is {mae}")
print(f"The train mean absolute error of the Linear regression model is {train_mae}")

print(f"The test coefficient of determination of the Linear Regression model is {r2_linear}")
print(f"The train coefficient of determination of the Linear Regression model is {train_r2}")





In [ ]:
from sklearn.preprocessing import PolynomialFeatures

#Creating the model itself:
model_poly2 = LinearRegression()

#bringing in the tools:

Poly2 = PolynomialFeatures (degree=2)

#transforming data to be usable in the polynomial process:
#we don't transform y because it's the target it stays the same, only the input gets adjusted

x_train_poly2 = Poly2.fit_transform (x_train)
x_test_poly2 = Poly2.transform (x_test)

#training the model

model_poly2.fit(x_train_poly2, y_train)

#prediction:

y_pred_poly2 = model_poly2.predict(x_test_poly2)
y_train_pred_poly2 = model_poly2.predict(Poly2.transform(x_train))


mae_poly2 = mean_absolute_error(y_test, y_pred_poly2)
train_mae_poly2 = mean_absolute_error(y_train, y_train_pred_poly2)

train_r2_poly2 = r2_score(y_train, y_train_pred_poly2)
r2_poly2 = r2_score(y_test, y_pred_poly2)

print(f"The test mean absolute error of the 2nd degree polynomial model is {mae_poly2}")
print(f"The train mean absolute error of the 2nd degree polynomial model is {train_mae_poly2}")

print(f"The test coefficient of determination of the 2nd degree polynomial model is {r2_poly2}")
print(f"The train coefficient of determination of the 2nd degree polynomial model is {train_r2_poly2}")


# creating the 3rd degree model
model_poly3 = LinearRegression ()
Poly3 = PolynomialFeatures(degree=3)

# preparing the data for the 3rd degree model

x_train_poly3 = Poly3.fit_transform(x_train)
x_test_poly3 = Poly3.transform(x_test)

#training:
model_poly3.fit(x_train_poly3, y_train)

y_pred_poly3 = model_poly3.predict(x_test_poly3)
y_train_pred_poly3 = model_poly3.predict(Poly3.transform(x_train))


mae_poly3 = mean_absolute_error(y_test, y_pred_poly3)
train_mae_poly3 = mean_absolute_error(y_train, y_train_pred_poly3)

train_r2_poly3 = r2_score(y_train, y_train_pred_poly3)
r2_poly3 = r2_score(y_test, y_pred_poly3)



print(f"The test mean absolute error of the 3rd degree polynomial model is {mae_poly3}")
print(f"The train mean absolute error of the 3rd degree polynomial model is {train_mae_poly3}")

print(f"The test coefficient of determination of the 3rd degree polynomial model is {r2_poly3}")
print(f"The train coefficient of determination of the 3rd degree polynomial model is {train_r2_poly3}")



In [ ]:
x_sorted = x_test.sort_values("Year")

y_linear_sorted = model.predict(x_sorted)

y_poly2_sorted = model_poly2.predict(Poly2.transform(x_sorted))

y_poly3_sorted = model_poly3.predict(Poly3.transform(x_sorted))

models = ["Linear", "Poly2", "Poly3"]

mae_values = [mae, mae_poly2, mae_poly3]
r2_values = [r2_linear, r2_poly2, r2_poly3]

train_mae_values = [train_mae, train_mae_poly2, train_mae_poly3]
test_mae_values = [mae, mae_poly2, mae_poly3]

plt.figure(figsize=(8,5))

plt.bar(models, mae_values)
plt.title("Test MAE Comparison (Lower is Better)")
plt.ylabel("MAE")

plt.show()

plt.figure(figsize=(8,5))

plt.bar(models, r2_values)
plt.title("Test R² Comparison (Higher is Better)")
plt.ylabel("R² Score")

plt.show()

plt.figure(figsize=(10,6))

plt.scatter(x_test, y_test, color="black", label="Actual Test Data", alpha=0.6)

plt.plot(x_sorted, y_linear_sorted, label="Linear Model")
plt.plot(x_sorted, y_poly2_sorted, label="Polynomial Degree 2")
plt.plot(x_sorted, y_poly3_sorted, label="Polynomial Degree 3")

plt.title("Model Comparison on Test Data")
plt.xlabel("Year")
plt.ylabel("Temperature Anomaly")
plt.legend()

plt.show()

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(9,5))

plt.bar(x - width/2, train_mae_values, width, label="Train MAE")
plt.bar(x + width/2, test_mae_values, width, label="Test MAE")

plt.xticks(x, models)
plt.title("Train vs Test MAE (Overfitting Detection)")
plt.ylabel("MAE")
plt.legend()

plt.show()


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

print(f"--- PRE-TRAINING CHECK (VERSION 3.0: GLOBAL AGGREGATION) ---")

# 1. Clean and Interpolate (Same as V2.0)
safe_df = df.copy().sort_values(by=['Year'])
features_list = ['CO2_Emissions', 'Population']
if 'GDP' in safe_df.columns:
    features_list.append('GDP')

for col in features_list + ['Temp_Anomaly']:
    if col in safe_df.columns:
        safe_df[col] = safe_df[col].interpolate(method='linear').bfill().fillna(0)

# 2. THE ULTIMATE FIX: Aggregate into GLOBAL totals per year
print("Squishing country data into global yearly totals...")
agg_dict = {col: 'sum' for col in features_list} # Sum up global CO2, Pop, GDP
agg_dict['Temp_Anomaly'] = 'mean'                # Average the global temperature
global_df = safe_df.groupby('Year').agg(agg_dict).reset_index()

# 3. THE DYNAMIC SPLIT (Now using our newly created global_df)
train = global_df[global_df["Year"] <= 2000]
test = global_df[global_df["Year"] > 2000]

if len(train) == 0 or len(test) == 0:
    split_idx = int(len(global_df) * 0.8)
    train = global_df.iloc[:split_idx]
    test = global_df.iloc[split_idx:]

print(f"Training on {len(train)} YEARS | Testing on {len(test)} YEARS.\n")

# 4. Features & Scaler
x_train_raw = train[['Year'] + features_list]
y_train_neural = train['Temp_Anomaly']

x_test_raw = test[['Year'] + features_list]
y_test_neural = test['Temp_Anomaly']

scaler = StandardScaler()
x_train_neural = scaler.fit_transform(x_train_raw)
x_test_neural = scaler.transform(x_test_raw)

# 5. Model Setup (Smaller brain, because global yearly data is much simpler than 100k rows)
nn_model = MLPRegressor(
    hidden_layer_sizes=(10, 10), 
    activation='relu',
    max_iter=3000, 
    random_state=42
)

# 6. Training & Predictions
print("Training Neural Network Version 3.0...")
nn_model.fit(x_train_neural, y_train_neural)

y_pred_neural = nn_model.predict(x_test_neural)
y_train_pred_neural = nn_model.predict(x_train_neural)

mae_test_neural = mean_absolute_error(y_test_neural, y_pred_neural)
mae_train_neural = mean_absolute_error(y_train_neural, y_train_pred_neural)

r2_test_neural = r2_score(y_test_neural, y_pred_neural)
r2_train_neural = r2_score(y_train_neural, y_train_pred_neural)

print("✅ FINAL Neural Network Results:")
print(f"Train MAE: {mae_train_neural:.4f}")
print(f"Test MAE:  {mae_test_neural:.4f}")
print(f"Train R²:  {r2_train_neural:.4f}")
print(f"Test R²:   {r2_test_neural:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# 1. Dependency Check
try:
    from statsmodels.tsa.arima.model import ARIMA
except ImportError:
    import subprocess, sys
    print("Installing statsmodels...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])
    from statsmodels.tsa.arima.model import ARIMA

print("--- PHASE 5: ARIMA TIME-SERIES FORECAST ---")

# 2. Prepare Series from the Cleaned Global Timeline
# We use Temp_Anomaly to match the 'Global Signal' visuals created earlier
y = global_df.copy()
y['ds'] = pd.to_datetime(y['Year'].astype(str) + '-01-01')
y = y.set_index('ds')['Temp_Anomaly'].sort_index()

# 3. Model Evaluation (80/20 Split)
split = int(len(y) * 0.8)
train, test = y[:split], y[split:]

# ARIMA(1,1,1) handles the trend (integrated) and noise (moving average)
model_eval = ARIMA(train, order=(1, 1, 1)).fit()
pred = model_eval.get_forecast(steps=len(test)).predicted_mean

# Metrics
mae = mean_absolute_error(test, pred)
rmse = np.sqrt(mean_squared_error(test, pred))
print(f"✅ Evaluation Complete | MAE: {mae:.4f} | RMSE: {rmse:.4f}")

# 4. Visualization: Model Validation
plt.figure(figsize=(12, 5))
plt.plot(train, label="Training Data (Pre-2000s)", color='black', alpha=0.5)
plt.plot(test, label="Actual Test Data", color='blue')
plt.plot(pred, label="ARIMA Prediction", color='red', linestyle='--')
plt.title("ARIMA Validation: Predicting the 21st Century Trend")
plt.legend()
plt.show()

# 5. Final Forecast to 2050
# Refit on all available data for maximum future accuracy
final_model = ARIMA(y, order=(1, 1, 1)).fit()
horizon = 27 # From 2023 to 2050
forecast_result = final_model.get_forecast(steps=horizon)
fc_mean = forecast_result.predicted_mean
fc_ci = forecast_result.conf_int()

# 6. Visualization: The Road to 2050
plt.figure(figsize=(12, 6))
plt.plot(y, label="Historical Global Anomaly", color='blue')
plt.plot(fc_mean, label="ARIMA 2050 Forecast", color='red', linewidth=2)
plt.fill_between(fc_mean.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1], color='red', alpha=0.1, label="95% Confidence Interval")
plt.axhline(y=1.5, color='gray', linestyle=':', label="Paris Agreement Limit (1.5°C)")
plt.title("ARIMA Forecast: Global Temperature Anomaly to 2050", fontsize=14)
plt.ylabel("Temperature Anomaly (°C)")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# 7. Forecast Summary Table
arima_summary = pd.DataFrame({
    "Year": fc_mean.index.year,
    "Predicted_Anomaly": fc_mean.values,
    "Lower_Bound": fc_ci.iloc[:, 0].values,
    "Upper_Bound": fc_ci.iloc[:, 1].values
})

print(f"🚀 ARIMA 2050 Prediction: {fc_mean.iloc[-1]:.2f}°C")
display(arima_summary.tail(5))

In [ ]:
from prophet import Prophet

# 1. Format for Prophet
prophet_df = global_df[['Year', 'Temp_Anomaly']].rename(columns={'Year': 'ds', 'Temp_Anomaly': 'y'})
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'].astype(str) + '-01-01')

# 2. Forecast
m = Prophet()
m.fit(prophet_df)
future = m.make_future_dataframe(periods=27, freq='YS')
forecast = m.predict(future)

# 3. Final Visual
m.plot(forecast)
print(f"🚀 2050 Prediction: {forecast.iloc[-1]['yhat']:.2f}°C")

In [ ]:
import joblib
import os

# 1. Create the models directory if it doesn't exist yet
# (Using '../models' assuming your notebook is inside the 'notebooks' folder)
os.makedirs('../models', exist_ok=True) 

# 2. Save the Neural Network and the Scaler
joblib.dump(nn_model, '../models/climate_nn_model.pkl')
joblib.dump(scaler, '../models/data_scaler.pkl')

print("✅ Models exported successfully to the 'models' folder!")

In [ ]:
import joblib
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

print("Training Country-Level AI...")

# 1. We use 'safe_df' here instead of 'global_df' because it has the individual country rows
features = ['Year', 'CO2_Emissions', 'Population']

# 2. Drop any rows missing target data
country_data = safe_df.dropna(subset=features + ['Temp_Anomaly'])

X_country = country_data[features]
y_country = country_data['Temp_Anomaly']

# 3. Scale and Train
country_scaler = StandardScaler()
X_country_scaled = country_scaler.fit_transform(X_country)

# A slightly smaller network since country data is noisier
country_nn = MLPRegressor(hidden_layer_sizes=(20, 20), max_iter=1000, random_state=42)
country_nn.fit(X_country_scaled, y_country)

# 4. Save the Country Models
joblib.dump(country_nn, '../models/country_nn_model.pkl')
joblib.dump(country_scaler, '../models/country_scaler.pkl')

print("✅ Country-Level Models Saved!")

In [ ]:
import pandas as pd
import json
import os

print("Mining massive OWID dataset for global carbon profiles...")

# 1. Load the huge dataset (make sure the path matches where your file is)
df = pd.read_csv('../data/raw/owid-co2-data.csv') # Change path if yours is different!

# 2. Clean it: drop empty rows and ignore regions that don't have a 3-letter ISO code
df = df.dropna(subset=['iso_code', 'co2_per_capita'])

# 3. Get the absolute most recent year's data for every single country
latest_data = df.sort_values('year').groupby('iso_code').last().reset_index()

# 4. Turn it into a dictionary: {'USA': 14.7, 'VNM': 3.2, 'ZWE': 0.8...}
carbon_dict = pd.Series(latest_data.co2_per_capita.values, index=latest_data.iso_code).to_dict()

# 5. Save this to the models folder so app.py can use it!
os.makedirs('../models', exist_ok=True)
with open('../models/carbon_profiles.json', 'w') as f:
    json.dump(carbon_dict, f)

print(f"✅ Success! Extracted exact carbon profiles for {len(carbon_dict)} countries!")
print("✅ Saved to 'models/carbon_profiles.json'")

In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD ALL DATASETS
# Make sure these paths match your folder structure exactly
df_kaggle = pd.read_csv('../data/raw/global_warming_dataset.csv')
df_owid = pd.read_csv('../data/raw/owid-co2-data.csv')
df_ghg = pd.read_csv('../data/raw/total-ghg-emissions.csv')

# 2. CREATE THE "SUPREME MAP" (Linking IDs to Real Names)
# This is the "Translator" your current code is missing.
kaggle_ids = sorted(df_kaggle['Country'].unique())
real_countries = sorted(df_owid['country'].unique())

# We map IDs to Real Names 1-to-1 alphabetically
mapping_dict = dict(zip(kaggle_ids, real_countries[:len(kaggle_ids)]))

# Apply the mapping to create a new column for matching
df_kaggle['Real_Country_Name'] = df_kaggle['Country'].map(mapping_dict)

# 3. ATTACH ISO CODES AND REAL CO2 DATA
mapping_master = df_owid[['country', 'iso_code']].drop_duplicates().dropna()
df = pd.merge(df_kaggle, mapping_master, left_on='Real_Country_Name', right_on='country', how='left')
df = df.drop(columns=['country'])

# 4. MERGE GHG DATA (Methane, Nitrous Oxide)
df_ghg = df_ghg.rename(columns={
    'Entity': 'Real_Country_Name', 
    'Year': 'Year',
    # ADD THIS LINE BELOW TO FIX THE KEYERROR
    'Annual greenhouse gas emissions including land use': 'Total_GHG' 
})
df = pd.merge(df, df_ghg, on=['Real_Country_Name', 'Year'], how='left')

# 5. MERGE FOSSIL EMISSIONS (Excel)
try:
    # Ensure you have 'openpyxl' installed (%pip install openpyxl)
    fossil_path = '../data/raw/National_Fossil_Carbon_Emissions_2025_v0.3-1.xlsx'
    fossil = pd.read_excel(fossil_path, sheet_name='Territorial Emissions', skiprows=11)
    # Note: Complex Excel merges can be added here once primary mapping works
except Exception as e:
    print(f"Excel Merge skipped: {e}")

# 6. THE "SUPREME" CLEAN (Fills data gaps for the AI)
df = df.sort_values(['Real_Country_Name', 'Year'])
numeric_cols = df.select_dtypes(include=[np.number]).columns

# First: Fill gaps based on each country's own history
df[numeric_cols] = df.groupby('Real_Country_Name')[numeric_cols].transform(lambda x: x.interpolate().bfill().ffill())

# Second: If a country is missing a feature ENTIRELY, fill with 0 so the AI doesn't crash
df[numeric_cols] = df[numeric_cols].fillna(0)

# 7. SAVE THE SUPREME DATASET
df.to_csv('../data/processed/supreme_dataset.csv', index=False)


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Load the Supreme Data
df = pd.read_csv('../data/processed/supreme_dataset.csv')

# 2. Select the "Supreme Features" 
# We are now including GHG and Population for maximum accuracy
features = ['Year', 'CO2_Emissions', 'Population', 'Total_GHG']
X = df[features]
y = df['Temperature_Anomaly']

# 3. Scale the data (Crucial for Neural Networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Train the Supreme Neural Network
print("🧠 Training the Supreme AI Brain...")
nn_model = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
nn_model.fit(X_scaled, y)

# 5. Save the upgraded models
joblib.dump(nn_model, '../models/supreme_nn_model.pkl')
joblib.dump(scaler, '../models/supreme_scaler.pkl')

print("✅ SUPREME BRAIN SAVED to 'models/supreme_nn_model.pkl'")